In [0]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("Spark window Function").getOrCreate()
import random
st_choices =['AP','WB','OD','MH']
data=[('01',"Santanu",1,1000,random.choice(st_choices)),
      ('02',"Synthia",2,1100,random.choice(st_choices)),
      ('03',"Suman",1,900,random.choice(st_choices)),
      ('04',"Sunny",4,800,random.choice(st_choices)),
      ('05',"Sunil",3,1000,random.choice(st_choices)),
      ('06',"Surya",2,1200,random.choice(st_choices)),
      ('07',"Suresh",2,1400,random.choice(st_choices)),
      ('08',"Suvashish",2,500,random.choice(st_choices))]
columns =['id','name','deptid','salary','State']
df_emp=spark.createDataFrame(data,columns)
df_emp.show()
dept_columns=['deptid','deptname']
dept_data=[(1,'Data Science'),(2,"DataEngineer"),(4,"Data Analyst")]
df_dept=spark.createDataFrame(dept_data,dept_columns)
df_dept.show()
df_new=df_emp.join(df_dept,df_dept.deptid==df_emp.deptid,'inner').select(df_emp.id,df_emp.name,df_emp.deptid,df_emp.salary,df_emp.State,df_dept.deptname)
df_new.show()

+------+------------+<br>
|deptid|    deptname|<br>
+------+------------+<br>
|     1|Data Science|<br>
|     2|DataEngineer|<br>
|     4|Data Analyst|<br>
+------+------------+<br>

+---+---------+------+------+-----+------------+<br>
| id|     name|deptid|salary|State|    deptname|<br>
+---+---------+------+------+-----+------------+<br>
| 01|  Santanu|     1|  1000|   WB|Data Science|<br>
| 02|  Synthia|     2|  1100|   AP|DataEngineer|<br>
| 03|    Suman|     1|   900|   WB|Data Science|<br>
| 04|    Sunny|     4|   800|   WB|Data Analyst|<br>
| 06|    Surya|     2|  1200|   OD|DataEngineer|<br>
| 07|   Suresh|     2|  1400|   AP|DataEngineer|<br>
| 08|Suvashish|     2|   500|   AP|DataEngineer|<br>
+---+---------+------+------+-----+------------+


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number,dense_rank,rank,col,lit
window_spec=Window.partitionBy(col("deptname")).orderBy(col("salary").desc())
deptwise_sal_spec=Window.partitionBy(col("deptname")).orderBy(col("salary").desc())
over_all_sal_spec=Window.partitionBy(lit("XX")).orderBy(col("salary").desc())
df_new.withColumn("Salary_rank_deptwise",row_number().over(window_spec))\
.withColumn("Salary_rank_overall",row_number().over(over_all_sal_spec)).show()

+---+---------+------+------+-----+------------+--------------------+-------------------+<br>
| id|     name|deptid|salary|State|    deptname|Salary_rank_deptwise|Salary_rank_overall|<br>
+---+---------+------+------+-----+------------+--------------------+-------------------+<br>
| 07|   Suresh|     2|  1400|   AP|DataEngineer|                   4|                  1|<br>
| 06|    Surya|     2|  1200|   OD|DataEngineer|                   3|                  2|<br>
| 02|  Synthia|     2|  1100|   AP|DataEngineer|                   2|                  3|<br>
| 01|  Santanu|     1|  1000|   WB|Data Science|                   2|                  4|<br>
| 03|    Suman|     1|   900|   WB|Data Science|                   1|                  5|<br>
| 04|    Sunny|     4|   800|   WB|Data Analyst|                   1|                  6|<br>
| 08|Suvashish|     2|   500|   AP|DataEngineer|                   1|                  7|<br>
+---+---------+------+------+-----+------------+--------------------+-------------------+